# llama-python-wrapper: High-Performance LLM Pipeline

This notebook demonstrates:
1. **Single-Session Inference**: Using the drop-in `Llama` class with managed background `llama-server.exe`.
2. **Multi-Session Concurrent Inference**: Using `LLMPool` to dispatch multiple independent sessions concurrently across parallel slots (`-np`) sharing model weights in GPU VRAM.
3. **100% Leak-Proof VRAM Reclamation**: Verifying that GPU VRAM is completely reclaimed upon unload.

In [1]:
from pathlib import Path
import time
import asyncio

from src import (
    Llama,
    LLM_Core,
    LLMPool,
    LLMConfig,
    GENConfig,
    GeneralLLMConfig,
    get_gpu_vram_info,
)

# Check GPU VRAM status
total_vram, free_vram = get_gpu_vram_info()
print(f"[Hardware] GPU VRAM: {free_vram:.0f} MB free / {total_vram:.0f} MB total")

# Find an available GGUF model in common paths
candidate_models = [
    Path("E:/Godseer/models/Qwen3.5-2B-Q4_K_M.gguf"),
    Path("E:/Godseer/models/Ministral-3-3B-Instruct-2512-Q5_K_M.gguf"),
    Path("E:/Godseer/models/Kev-4B-Q4_K_M.gguf"),
    Path("./models/model.gguf"),
]

MODEL_PATH = None
for p in candidate_models:
    if p.exists():
        MODEL_PATH = str(p)
        break

if MODEL_PATH:
    print(f"[Selected Model] {MODEL_PATH}")
else:
    raise FileNotFoundError("No GGUF model found in candidate paths. Please set MODEL_PATH.")

[Hardware] GPU VRAM: 7390 MB free / 8188 MB total
[Selected Model] E:\Godseer\models\Qwen3.5-2B-Q4_K_M.gguf


## 1. Single Session Inference (`Llama`)

The `Llama` class is a drop-in replacement for `llama_cpp.Llama`. It transparently starts a supervised native `llama-server.exe` process in the background and sends requests over HTTP.

In [2]:
print("--- Starting Single Session Inference ---")

with Llama(model_path=MODEL_PATH, n_ctx=2048, verbose=False) as llm:
    messages = [
        {"role": "system", "content": "You are a concise AI assistant."},
        {"role": "user", "content": "What are the three primary colors of light in physics? Answer in one sentence."},
    ]
    
    t0 = time.perf_counter()
    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=64,
        temperature=0.2,
    )
    elapsed = time.perf_counter() - t0
    
    msg = response["choices"][0]["message"]
    content = msg.get("content") or msg.get("reasoning_content") or ""
    usage = response.get("usage", {})
    timings = response.get("timings", {})
    tok_per_sec = timings.get("predicted_per_second", 0.0)
    
    print(f"\n[Prompt]: {messages[-1]['content']}")
    print(f"[Response]:\n{content.strip()}")
    print(f"\n[Metrics]: Latency: {elapsed:.2f}s | Tokens: {usage.get('completion_tokens', 0)} out | Speed: {tok_per_sec:.1f} tok/s")

# Check VRAM after unload
_, free_after = get_gpu_vram_info()
print(f"\n[VRAM Verification] Free VRAM after unload: {free_after:.0f} MB (Reclaimed successfully)")

--- Starting Single Session Inference ---


llama_server.py][calculate_optimal_concurrency] - [VRAM Optimizer] Validated requested n_parallel=1 against model 'Qwen3.5-2B-Q4_K_M.gguf' (1222MB, 125MB/slot). Safe capacity: 6 slots.
llama_server.py][start] - [LlamaServer] Launching Qwen3.5-2B-Q4_K_M.gguf on http://127.0.0.1:8080 (-np 1, ctx/slot: 2048, -fa 1)...
llama_server.py][start] - [LlamaServer] Ready in 2.04s.



[Prompt]: What are the three primary colors of light in physics? Answer in one sentence.
[Response]:
Thinking Process:

1.  **Analyze the Request:**
    *   Question: "What are the three primary colors of light in physics?"
    *   Constraint: "Answer in one sentence."

2.  **Identify the Core Fact:**
    *   The three primary colors of *

[Metrics]: Latency: 0.63s | Tokens: 64 out | Speed: 113.5 tok/s


llama_server.py][stop] - [LlamaServer] Stopped PID 3164 & VRAM reclaimed in 0.190s.



[VRAM Verification] Free VRAM after unload: 7383 MB (Reclaimed successfully)


## 2. Multi-Session Concurrent Inference (`LLMPool`)

`LLMPool` runs continuous batching across parallel slots (`-np`) on a single model in VRAM.
Here we dispatch multiple independent user sessions simultaneously using `asyncio`.

In [3]:
print("--- Starting Multi-Session Concurrent Inference ---")

NUM_SESSIONS = 3

# Create pool with 3 parallel slots
pool = LLMPool(
    LLM_Core,
    n_workers=NUM_SESSIONS,
    general_config=GeneralLLMConfig(model_path=MODEL_PATH),
    llm_config=LLMConfig(n_ctx=2048),
)

pool.load()

session_prompts = [
    "Explain quantum superposition in 2 sentences.",
    "Solve this math riddle: What is (15 * 4) + (100 / 5)?",
    "Name 3 planets in the solar system known to have rings.",
]

async def run_parallel_sessions():
    t_start = time.perf_counter()
    tasks = [
        pool.call("chat", prompt=prompt, sys_prompt="You are a helpful and concise assistant.")
        for prompt in session_prompts
    ]
    
    responses = await asyncio.gather(*tasks)
    total_time = time.perf_counter() - t_start
    
    print(f"\n=== Processed {len(responses)} Concurrent Sessions in {total_time:.2f}s ===\n")
    for i, (prompt, response) in enumerate(zip(session_prompts, responses), 1):
        print(f"[Session {i} Prompt]: {prompt}")
        print(f"[Session {i} Reply]:\n{response.strip()}\n")
        print("-" * 50)

# Execute concurrent sessions
await run_parallel_sessions()

# Unload pool and reclaim GPU resources
pool.unload()

_, free_final = get_gpu_vram_info()
print(f"\n[Final VRAM] Free VRAM after pool unload: {free_final:.0f} MB (100% reclaimed)")

--- Starting Multi-Session Concurrent Inference ---


llmRouter.py][__init__] - [LLM_Core] Initialized with model: Qwen3.5-2B-Q4_K_M.gguf
llmRouter.py][__init__] - [LLM_Core] LLM Config: {
  "n_gpu_layers": -1,
  "n_ctx": 2048,
  "n_batch": 1024,
  "n_threads": 8,
  "verbose": false,
  "flash_attn": true,
  "seed": 42,
  "offload_kqv": true,
  "logits_all": false,
  "type_k": 2,
  "type_v": 2,
  "n_ubatch": 512,
  "use_mlock": true
}
llmRouter.py][__init__] - [LLM_Core] Generation Config: {
  "max_tokens": 8000,
  "temperature": 0.2,
  "top_k": 50,
  "top_p": 0.9,
  "repeat_penalty": 1.1,
  "stop": []
}
llmRouter.py][_load_local_model] - [LLM_Core] Loading model: Qwen3.5-2B-Q4_K_M.gguf
llama_server.py][calculate_optimal_concurrency] - [VRAM Optimizer] Validated requested n_parallel=3 against model 'Qwen3.5-2B-Q4_K_M.gguf' (1222MB, 125MB/slot). Safe capacity: 6 slots.
llama_server.py][start] - [LlamaServer] Launching Qwen3.5-2B-Q4_K_M.gguf on http://127.0.0.1:8080 (-np 3, ctx/slot: 2048, -fa 1)...


[PARALLEL MODE] Model 'Qwen3.5-2B-Q4_K_M.gguf' running with 3 parallel slots.


llama_server.py][start] - [LlamaServer] Ready in 1.83s.
llmRouter.py][_load_local_model] - [LLM_Core] Ready in 2.2s
llmCore.py][chat] - ###### Prompt: 
	[{'role': 'system', 'content': 'You are a helpful and concise assistant.'}, {'role': 'user', 'content': 'Explain quantum superposition in 2 sentences.'}]

llmCore.py][chat] - ###### Prompt: 
	[{'role': 'system', 'content': 'You are a helpful and concise assistant.'}, {'role': 'user', 'content': 'Solve this math riddle: What is (15 * 4) + (100 / 5)?'}]

llmCore.py][chat] - ###### Prompt: 
	[{'role': 'system', 'content': 'You are a helpful and concise assistant.'}, {'role': 'user', 'content': 'Name 3 planets in the solar system known to have rings.'}]

llmCore.py][chat] - ###### Active Schema: 
	{}

llmCore.py][chat] - ###### Active Schema: 
	{}

llmCore.py][chat] - ###### Active Schema: 
	{}

llmCore.py][chat] -  [8.8s | 745 tok | local/Qwen3.5-2B-Q4_K_M.gguf] 
llmCore.py][chat] - PROMPT TOK: 48
llmCore.py][chat] - TOTAL TOK: 793
llmCor


=== Processed 3 Concurrent Sessions in 22.04s ===

[Session 1 Prompt]: Explain quantum superposition in 2 sentences.
[Session 1 Reply]:
Quantum superposition is a fundamental principle of quantum mechanics where a physical system can exist in multiple states simultaneously until it is measured. Upon measurement, this wave function collapses into a single definite state, eliminating the uncertainty associated with all other possibilities.

--------------------------------------------------
[Session 2 Prompt]: Solve this math riddle: What is (15 * 4) + (100 / 5)?
[Session 2 Reply]:
The answer is **80**.

Here is the breakdown:
1.  $15 \times 4 = 60$
2.  $100 / 5 = 20$
3.  $60 + 20 = 80$

--------------------------------------------------
[Session 3 Prompt]: Name 3 planets in the solar system known to have rings.
[Session 3 Reply]:
Three planets known to have ring systems include **Saturn**, **Jupiter**, and **Neptune**. (Uranus also has a prominent ring system).

-----------------------

llmCore.py][analyze_token_usage] - ==================================================
llmCore.py][analyze_token_usage] - [LLM_Core] Token usage analysis:
llmCore.py][analyze_token_usage] - Total prompt tokens: 117
llmCore.py][analyze_token_usage] - Total completion tokens: 3444
llmCore.py][analyze_token_usage] - Total tokens: 3561
llmCore.py][analyze_token_usage] - Max prompt tokens in a single request: 48
llmCore.py][analyze_token_usage] - Max completion tokens in a single request: 1593
llmCore.py][analyze_token_usage] - Max total tokens in a single request: 1626
llmCore.py][analyze_token_usage] - Total entries processed: 3
llmCore.py][analyze_token_usage] - ==================================================
llmCore.py][analyze_token_usage] - Avg. prompt tokens per entry: 39.00
llmCore.py][analyze_token_usage] - Avg. completion tokens per entry: 1148.00
llmCore.py][analyze_token_usage] - Avg. total tokens per entry: 1187.00
llmCore.py][analyze_token_usage] - ==========================


[Final VRAM] Free VRAM after pool unload: 7376 MB (100% reclaimed)
